In [1]:
import gzip
import re
from collections import Counter, defaultdict
from pathlib import Path

import polars as pl

DATA = Path("../../skos")

cfg = pl.Config(tbl_rows=-1, tbl_cols=-1, fmt_str_lengths=10000, tbl_width_chars=10000, tbl_hide_dataframe_shape=True)

In [2]:
PREFIXES = {
    "rdf": "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
    "rdfs": "http://www.w3.org/2000/01/rdf-schema#",
    "owl": "http://www.w3.org/2002/07/owl#",
    "xsd": "http://www.w3.org/2001/XMLSchema#",
    "skos": "http://www.w3.org/2004/02/skos/core#",
    "skosxl": "http://www.w3.org/2008/05/skos-xl#",
    "dcterms": "http://purl.org/dc/terms/",
    "dc": "http://purl.org/dc/elements/1.1/",
    "foaf": "http://xmlns.com/foaf/0.1/",
    "dcat": "http://www.w3.org/ns/dcat#",
    "prov": "http://www.w3.org/ns/prov#",
    "vann": "http://purl.org/vocab/vann/",
    "void": "http://rdfs.org/ns/void#",
    "schema": "http://schema.org/",
    "euvoc": "http://publications.europa.eu/ontology/euvoc#",
    "at": "http://publications.europa.eu/ontology/authority/",
    "gn": "https://www.geonames.org/ontology#",
    "zbwext": "http://zbw.eu/namespaces/zbw-extensions/",
    "unesco": "http://vocabularies.unesco.org/ontology#",
    "gemet": "http://www.eionet.europa.eu/gemet/2004/06/gemet-schema.rdf#",
    "physh": "https://physh.org/rdf/2018/01/01/core#",
    "locn": "https://www.w3.org/ns/locn#",
    "cs": "http://purl.org/vocab/changeset/schema#",
    "iso-thes": "http://purl.org/iso25964/skos-thes#",
    "lexinfo": "http://www.lexinfo.net/ontology/2.0/lexinfo#",
    "xkos": "http://rdf-vocabulary.ddialliance.org/xkos#",
}
RDF_TYPE = "<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>"
SUBCLASS_OF = "<http://www.w3.org/2000/01/rdf-schema#subClassOf>"
TERM = re.compile(r'<[^>]*>|_:\S+|"(?:[^"\\]|\\.)*"(?:@[\w-]+|\^\^<[^>]*>)?')
LITERAL = re.compile(r'"(?:[^"\\]|\\.)*"(?:@([\w-]+)|\^\^(<[^>]*>))?')


def short(term):
    if not term.startswith("<"):
        return term
    iri = term[1:-1]
    for prefix, ns in PREFIXES.items():
        if iri.startswith(ns):
            return f"{prefix}:{iri[len(ns):]}"
    return term


def literal_type(term):
    lang, dt = LITERAL.fullmatch(term).groups()
    return "rdf:langString" if lang else short(dt) if dt else "xsd:string"


def triples(path):
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                yield tuple(TERM.findall(line)[:3])

In [3]:
def stats(path):
    ts = set(triples(path))
    types = defaultdict(set)
    for s, p, o in ts:
        if p == RDF_TYPE:
            types[s].add(o)
    class_uses = Counter(o for _, p, o in ts if p == RDF_TYPE)
    supers = defaultdict(set)
    for s, p, o in ts:
        if p == SUBCLASS_OF and s in class_uses:
            supers[s].add(o)
    untyped = {"(untyped)"}
    prop_uses, domains, ranges = Counter(), defaultdict(set), defaultdict(set)
    for s, p, o in ts:
        prop_uses[p] += 1
        domains[p] |= types.get(s, untyped)
        if o.startswith('"'):
            ranges[p].add(literal_type(o))
        else:
            ranges[p] |= types.get(o, untyped)
    names = lambda xs: sorted(short(x) for x in xs)
    classes = [
        {"class": short(c), "uses": n, "subClassOf": names(supers[c])}
        for c, n in class_uses.items()
    ]
    props = [
        {"property": short(p), "uses": n, "domains": names(domains[p]), "ranges": names(ranges[p])}
        for p, n in prop_uses.items()
    ]
    return len(ts), classes, props


datasets = sorted(d.name for d in DATA.iterdir() if (d / "main.nt.gz").exists())
overview, class_rows, prop_rows = [], [], []
for name in datasets:
    n, classes, props = stats(DATA / name / "main.nt.gz")
    overview.append({"dataset": name, "triples": n, "classes": len(classes), "properties": len(props)})
    class_rows += [{"dataset": name, **r} for r in classes]
    prop_rows += [{"dataset": name, **r} for r in props]

overview = pl.DataFrame(overview)
classes = pl.DataFrame(class_rows, schema_overrides={"subClassOf": pl.List(pl.String)})
props = pl.DataFrame(prop_rows)

In [4]:
def joined(df):
    return df.with_columns(pl.col(pl.List(pl.String)).list.join(", "))


overview.sort("triples", descending=True)

dataset,triples,classes,properties
str,i64,i64,i64
"""eu-nuts-2021""",383781,11,65
"""gemet""",323635,7,34
"""eu-cpv""",264759,2,7
"""stw-economics""",120694,15,65
"""eu-nace-rev2""",120439,7,53
"""eu-languages""",116190,3,21
"""unesco-thesaurus""",99711,6,38
"""loc-genre-form""",76520,3,18
"""nerc-p06-units""",65529,2,33


In [5]:
joined(
    classes.group_by("class")
    .agg(
        pl.col("uses").sum(),
        pl.col("dataset").n_unique().alias("datasets"),
        pl.col("subClassOf").list.explode(keep_nulls=False, empty_as_null=False).unique().sort(),
    )
    .sort("uses", descending=True)
)

class,uses,datasets,subClassOf
str,i64,u32,str
"""skos:Concept""",49853,16,""""""
"""locn:Geometry""",27660,1,""""""
"""dcat:Distribution""",27660,1,""""""
"""at:MappedCode""",9325,1,""""""
"""skosxl:Label""",9073,3,""""""
"""zbwext:Descriptor""",7253,1,"""skos:Concept"""
"""xkos:ConceptAssociation""",6606,2,""""""
"""cs:ChangeSet""",5740,1,""""""
"""euvoc:XlNotation""",2485,1,""""""


In [6]:
for name in datasets:
    print(name)
    display(joined(classes.filter(pl.col("dataset") == name).drop("dataset").sort("uses", descending=True)))

coar-resource-types


class,uses,subClassOf
str,i64,str
"""skos:Concept""",110,""""""
"""skos:ConceptScheme""",1,""""""
"""skos:Collection""",1,""""""


eu-cpv


class,uses,subClassOf
str,i64,str
"""skos:Concept""",9454,""""""
"""skos:ConceptScheme""",1,""""""


eu-data-theme


...

class,uses,subClassOf
str,i64,str
"""skos:Concept""",7845,""""""
"""zbwext:Descriptor""",7253,"""skos:Concept"""
"""zbwext:Thsys""",592,"""skos:Concept"""
"""rdf:Property""",28,""""""
"""owl:ObjectProperty""",25,""""""
"""owl:AnnotationProperty""",13,""""""
"""owl:Class""",13,""""""
"""owl:FunctionalProperty""",7,""""""
"""owl:SymmetricProperty""",4,""""""


uat-astronomy


class,uses,subClassOf
str,i64,str
"""skos:Concept""",2411,""""""
"""skos:ConceptScheme""",1,""""""


unesco-thesaurus


class,uses,subClassOf
str,i64,str
"""skos:Concept""",4500,""""""
"""iso-thes:ConceptGroup""",95,""""""
"""skos:Collection""",95,""""""
"""unesco:MicroThesaurus""",88,""""""
"""unesco:Domain""",7,""""""
"""skos:ConceptScheme""",1,""""""


In [7]:
joined(
    props.group_by("property")
    .agg(
        pl.col("uses").sum(),
        pl.col("dataset").n_unique().alias("datasets"),
        pl.col("domains").list.explode(keep_nulls=False, empty_as_null=False).unique().sort(),
        pl.col("ranges").list.explode(keep_nulls=False, empty_as_null=False).unique().sort(),
    )
    .sort("uses", descending=True)
)

property,uses,datasets,domains,ranges
str,i64,u32,str,str
"""skos:prefLabel""",525591,17,"""dcat:Catalog, dcat:Dataset, gemet:Group, gemet:SuperGroup, gn:Code, iso-thes:ConceptGroup, physh:Concept, physh:Facet, skos:Collection, skos:Concept, skos:ConceptScheme, skos:OrderedCollection, unesco:Domain, unesco:MicroThesaurus, void:Dataset, xkos:ClassificationLevel, xkos:Correspondence, zbwext:Descriptor, zbwext:Thsys""","""rdf:langString, xsd:string"""
"""rdf:type""",148449,17,"""<http://www.w3.org/2006/vcard/ns#Kind>, at:MappedCode, cs:ChangeSet, dcat:Catalog, dcat:Dataset, dcat:Distribution, euvoc:XlNotation, foaf:Agent, foaf:Organization, foaf:Person, gemet:Group, gemet:Source, gemet:SuperGroup, gemet:Theme, gn:Class, gn:Code, iso-thes:ConceptGroup, locn:Geometry, owl:AnnotationProperty, owl:Class, owl:DatatypeProperty, owl:FunctionalProperty, owl:InverseFunctionalProperty, owl:ObjectProperty, owl:Ontology, owl:Restriction, owl:SymmetricProperty, owl:TransitiveProperty, physh:Concept, physh:Discipline, physh:Facet, rdf:Property, skos:Collection, skos:Concept, skos:ConceptScheme, skos:OrderedCollection, skosxl:Label, unesco:Domain, unesco:MicroThesaurus, void:Dataset, xkos:ClassificationLevel, xkos:ConceptAssociation, xkos:Correspondence, zbwext:Descriptor, zbwext:Thsys""","""(untyped), owl:Class"""
"""skos:related""",90276,7,"""physh:Concept, skos:Concept, zbwext:Descriptor""","""(untyped), physh:Concept, skos:Concept, zbwext:Descriptor"""
"""skos:altLabel""",83182,11,"""(untyped), iso-thes:ConceptGroup, physh:Concept, skos:Collection, skos:Concept, skos:ConceptScheme, unesco:MicroThesaurus, zbwext:Descriptor""","""rdf:langString, xsd:string"""
"""skos:note""",57036,5,"""dcat:Distribution, skos:Concept, skos:ConceptScheme, skosxl:Label""","""rdf:langString, xsd:string"""
"""skos:inScheme""",51235,15,"""(untyped), dcat:Catalog, dcat:Dataset, gemet:Group, gemet:SuperGroup, gemet:Theme, gn:Code, iso-thes:ConceptGroup, skos:Collection, skos:Concept, skos:ConceptScheme, unesco:Domain, unesco:MicroThesaurus, zbwext:Descriptor, zbwext:Thsys""","""(untyped), dcat:Catalog, dcat:Dataset, gn:Class, skos:ConceptScheme, void:Dataset"""
"""skos:definition""",50710,10,"""dcat:Catalog, dcat:Dataset, gn:Code, owl:AnnotationProperty, owl:Class, owl:DatatypeProperty, owl:FunctionalProperty, owl:InverseFunctionalProperty, owl:ObjectProperty, owl:SymmetricProperty, owl:TransitiveProperty, rdf:Property, skos:Concept, skos:ConceptScheme""","""(untyped), rdf:langString, xsd:string"""
"""skos:broader""",48841,13,"""physh:Concept, physh:Facet, skos:Concept, zbwext:Descriptor, zbwext:Thsys""","""(untyped), physh:Concept, physh:Facet, skos:Concept, zbwext:Descriptor, zbwext:Thsys"""
"""skos:narrower""",38212,10,"""physh:Concept, physh:Facet, skos:Concept, zbwext:Descriptor, zbwext:Thsys""","""physh:Concept, physh:Facet, skos:Concept, zbwext:Descriptor, zbwext:Thsys"""


In [8]:
for name in datasets:
    print(name)
    display(joined(props.filter(pl.col("dataset") == name).drop("dataset").sort("uses", descending=True)))

coar-resource-types


property,uses,domains,ranges
str,i64,str,str
"""skos:prefLabel""",2750,"""skos:Collection, skos:Concept""","""rdf:langString"""
"""skos:altLabel""",1107,"""skos:Concept""","""rdf:langString"""
"""rdf:type""",112,"""skos:Collection, skos:Concept, skos:ConceptScheme""","""(untyped)"""
"""skos:definition""",111,"""skos:Concept""","""rdf:langString"""
"""skos:inScheme""",110,"""skos:Concept""","""skos:ConceptScheme"""
"""skos:broadMatch""",91,"""skos:Concept""","""(untyped), skos:Concept"""
"""skos:exactMatch""",88,"""skos:Concept""","""(untyped)"""
"""skos:narrower""",84,"""skos:Concept""","""skos:Concept"""
"""skos:broader""",84,"""skos:Concept""","""skos:Concept"""


eu-cpv


property,uses,domains,ranges
str,i64,str,str
"""skos:prefLabel""",226897,"""skos:Concept, skos:ConceptScheme""","""rdf:langString"""
"""rdf:type""",9455,"""skos:Concept, skos:ConceptScheme""","""(untyped)"""
"""skos:inScheme""",9454,"""skos:Concept""","""skos:ConceptScheme"""
"""skos:notation""",9454,"""skos:Concept""","""xsd:string"""
"""skos:broader""",9409,"""skos:Concept""","""skos:Concept"""
"""skos:topConceptOf""",45,"""skos:Concept""","""skos:ConceptScheme"""
"""skos:hasTopConcept""",45,"""skos:ConceptScheme""","""skos:Concept"""


eu-data-theme


...

property,uses,domains,ranges
str,i64,str,str
"""skos:altLabel""",20451,"""skos:Concept, zbwext:Descriptor""","""rdf:langString"""
"""rdf:type""",15788,"""owl:AnnotationProperty, owl:Class, owl:DatatypeProperty, owl:FunctionalProperty, owl:InverseFunctionalProperty, owl:ObjectProperty, owl:Ontology, owl:SymmetricProperty, owl:TransitiveProperty, rdf:Property, skos:Concept, skos:ConceptScheme, void:Dataset, zbwext:Descriptor, zbwext:Thsys""","""(untyped), owl:Class"""
"""skos:related""",14522,"""skos:Concept, zbwext:Descriptor""","""skos:Concept, zbwext:Descriptor"""
"""skos:narrower""",14351,"""skos:Concept, zbwext:Descriptor, zbwext:Thsys""","""skos:Concept, zbwext:Descriptor, zbwext:Thsys"""
"""skos:broader""",14351,"""skos:Concept, zbwext:Descriptor, zbwext:Thsys""","""skos:Concept, zbwext:Descriptor, zbwext:Thsys"""
"""skos:prefLabel""",13394,"""skos:Concept, skos:ConceptScheme, void:Dataset, zbwext:Descriptor, zbwext:Thsys""","""rdf:langString"""
"""skos:inScheme""",7845,"""skos:Concept, zbwext:Descriptor, zbwext:Thsys""","""skos:ConceptScheme, void:Dataset"""
"""<http://purl.org/ontology/gbv/gvkppn>""",6198,"""skos:Concept, zbwext:Descriptor""","""xsd:string"""
"""rdfs:label""",3352,"""owl:AnnotationProperty, owl:Class, owl:DatatypeProperty, owl:FunctionalProperty, owl:InverseFunctionalProperty, owl:ObjectProperty, owl:SymmetricProperty, owl:TransitiveProperty, rdf:Property, skos:Concept, zbwext:Descriptor, zbwext:Thsys""","""rdf:langString"""


uat-astronomy


property,uses,domains,ranges
str,i64,str,str
"""skos:narrower""",2693,"""skos:Concept""","""skos:Concept"""
"""skos:broader""",2693,"""skos:Concept""","""skos:Concept"""
"""dcterms:modified""",2414,"""skos:Concept, skos:ConceptScheme""","""xsd:dateTime"""
"""rdf:type""",2412,"""skos:Concept, skos:ConceptScheme""","""(untyped)"""
"""skos:prefLabel""",2354,"""skos:Concept""","""rdf:langString"""
"""skos:altLabel""",1839,"""skos:Concept""","""rdf:langString"""
"""dcterms:contributor""",1683,"""skos:Concept, skos:ConceptScheme""","""(untyped)"""
"""dcterms:created""",1655,"""(untyped), skos:Concept""","""xsd:dateTime"""
"""dcterms:creator""",1655,"""(untyped), skos:Concept""","""(untyped)"""


unesco-thesaurus


property,uses,domains,ranges
str,i64,str,str
"""skos:prefLabel""",22939,"""iso-thes:ConceptGroup, skos:Collection, skos:Concept, skos:ConceptScheme, unesco:Domain, unesco:MicroThesaurus""","""rdf:langString"""
"""skos:altLabel""",21071,"""iso-thes:ConceptGroup, skos:Collection, skos:Concept, unesco:MicroThesaurus""","""rdf:langString"""
"""skos:related""",12583,"""skos:Concept""","""skos:Concept"""
"""skos:hiddenLabel""",11470,"""iso-thes:ConceptGroup, skos:Collection, skos:Concept, unesco:Domain, unesco:MicroThesaurus""","""rdf:langString"""
"""rdf:type""",4786,"""iso-thes:ConceptGroup, skos:Collection, skos:Concept, skos:ConceptScheme, unesco:Domain, unesco:MicroThesaurus""","""(untyped)"""
"""dcterms:modified""",4597,"""(untyped), iso-thes:ConceptGroup, skos:Collection, skos:Concept, skos:ConceptScheme, unesco:Domain, unesco:MicroThesaurus""","""xsd:date, xsd:dateTime"""
"""skos:inScheme""",4595,"""iso-thes:ConceptGroup, skos:Collection, skos:Concept, unesco:Domain, unesco:MicroThesaurus""","""skos:ConceptScheme"""
"""skos:member""",4588,"""iso-thes:ConceptGroup, skos:Collection, unesco:Domain, unesco:MicroThesaurus""","""iso-thes:ConceptGroup, skos:Collection, skos:Concept, unesco:MicroThesaurus"""
"""skos:narrower""",4343,"""skos:Concept""","""skos:Concept"""
